# 36 — Skill Power Tools And Metadata

This notebook shows how skills now do two jobs at runtime:

- they improve the agent's prompt and workflow
- they can automatically attach stronger built-in tools for that run

It also shows how to inspect `used_skills` and `used_skill_tools` from the final `AgentResult.metadata`.

In [ ]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.run_multi_tool_agent import build_llm_from_env
from shipit_agent import Agent, DeepAgent, DEFAULT_SKILLS_PATH, FileSkillRegistry

llm = build_llm_from_env("bedrock")

## 1. Inspect The Packaged Skills Catalog

In [ ]:
registry = FileSkillRegistry(DEFAULT_SKILLS_PATH)
print("Catalog path:", DEFAULT_SKILLS_PATH)
print("Total packaged skills:", len(registry.list()))

for skill in registry.search("workflow")[:5]:
    print(f"- {skill.id}: {skill.description}")

## 2. `skills` vs `auto_use_skills`

- `skills=[...]` means these skills are always attached
- `auto_use_skills=True` means the agent may also auto-match more skills from the prompt
- `project_root="/tmp"` is now the default for built-in project tools, but it is shown explicitly here for clarity

In [ ]:
agent = Agent.with_builtins(
    llm=llm,
    project_root="/tmp",
    skills=["code-workflow-assistant"],
    auto_use_skills=True,
    skill_match_limit=2,
)

print("Explicit skills attached:")
for skill in agent.skills:
    print("-", skill.id)

## 3. Use A Skill That Pulls In Stronger Tools

The `web-scraper-pro` skill can now bring in tools like:

- `web_search`
- `open_url`
- `playwright_browse`
- `bash`
- `read_file`
- `edit_file`
- `write_file`
- `glob_files`
- `grep_files`

In [ ]:
scraper_agent = Agent.with_builtins(
    llm=llm,
    project_root="/tmp",
    skills=["web-scraper-pro", "code-workflow-assistant"],
    auto_use_skills=False,
)

tool_names = sorted(
    tool.name for tool in scraper_agent._effective_tools("scrape and patch the parser")
)
for name in tool_names:
    print(name)

## 4. Run The Agent And Inspect Metadata

The result metadata now reports which skills were active and which extra built-in tools were injected because of those skills.

In [ ]:
prompt = """
We have local work under /tmp.

Please:
1. inspect the scraper code
2. patch the output format if needed
3. save the cleaned result
"""

result = scraper_agent.run(prompt)

print("Used skills:", result.metadata.get("used_skills"))
print("Used skill tools:", result.metadata.get("used_skill_tools"))
print("\nModel output:\n")
print(result.output)

## 5. DeepAgent Works The Same Way

In [ ]:
deep_agent = DeepAgent.with_builtins(
    llm=llm,
    project_root="/tmp",
    skills=["web-scraper-pro"],
    default_skill_ids=["code-workflow-assistant"],
    auto_use_skills=True,
    skill_match_limit=2,
)

deep_result = deep_agent.run(
    "Inspect the local scraper under /tmp, fix the slow path, and summarize the changes."
)

print("Used skills:", deep_result.metadata.get("used_skills"))
print("Used skill tools:", deep_result.metadata.get("used_skill_tools"))
print("\nDeepAgent output:\n")
print(getattr(deep_result, "output", str(deep_result)))

## 6. Notes

- Skills improve both reasoning and tool availability
- `skills=[...]` is deterministic
- `auto_use_skills=True` is adaptive
- `project_root` controls where the project file tools and `bash` operate
- `used_skills` and `used_skill_tools` make the final run easier to inspect